In [4]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [5]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [6]:
tokenizer_path = TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer")
tokenizer = tokenizerLib.Tokenizer.load(tokenizer_path)

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json


In [7]:
dataset = svg_dataset.SVGDataset(
    OUR_DATASET_DIRECTORY, context_size=4096,
    tokenizer=tokenizer.encode, decoder=tokenizer.decode)

In [8]:
device_type = "cuda"
device_type = (autodetect_device_type() if device_type == "" else device_type)
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = \
    compute_init(device_type)

2026-03-08 15:16:16,152 - LLM.nanochat.common - INFO - Distributed world size: 1


In [9]:
aspect_ratio = 10.5
head_dim = 128
max_seq_len = 2048*8
window_pattern = "SSSL"
vocab_size = 2048


def build_model_meta(depth):
    """Build a model on meta device for a given depth (shapes/dtypes only, no data)."""
    # Model dim is nudged up to nearest multiple of head_dim for clean division
    # (FA3 requires head_dim divisible by 8, and this guarantees head_dim == args.head_dim exactly)
    base_dim = depth * aspect_ratio
    model_dim = int(((base_dim + head_dim - 1) // head_dim) * head_dim)
    num_heads = int(model_dim // head_dim)
    print(f"num heads: {model_dim / head_dim}")
    config = nanoChatModel.GPTConfig(
        sequence_len=max_seq_len, vocab_size=vocab_size,
        n_layer=depth, n_head=num_heads, n_kv_head=num_heads, n_embd=model_dim,
        window_pattern=window_pattern)
    print(config)
    with torch.device("meta"):
        model_meta = nanoChatModel.GPT(config)
    return model_meta

model = Model()

model.llm = build_model_meta(6)
# 2) All tensors get storage on target device but with uninitialized (garbage) data
model.llm.to_empty(device=device)
model.llm.init_weights()  # 3) All tensors get initialized
model.optimizer = model.llm.setup_optimizer()

params = model.llm.num_scaling_params()
print(params.items())
params_Embed = (params['wte'] + params['value_embeds'])
print(f"{params['total']:_d} params "
      f"(with embeding: {params_Embed:_d} | "
      f"last layer: {params['lm_head']:_d} | "
      f"transformer: {params['transformer_matrices']:_d})")
model.llm = model.llm.bfloat16()

# the inputs to model will never change shape so dynamic=False is safe
model.llm = torch.compile(model.llm, dynamic=False) # type: ignore
# model.eval();


num heads: 1.0
GPTConfig(sequence_len=16384, vocab_size=2048, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
dict_items([('wte', 262144), ('value_embeds', 786432), ('lm_head', 262144), ('transformer_matrices', 1179744), ('scalars', 12), ('total', 2490476)])
2_490_476 params (with embeding: 1_048_576 | last layer: 262_144 | transformer: 1_179_744)


In [82]:
import importlib
import metrics.metrics
_ = importlib.reload(metrics.metrics)
import metrics.metrics

In [85]:

testDevice = torch.device("cuda:0")

prof = Profiler(["iterDataloader", "split", "target", "logits", "print"])

dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)
accum = metrics.metrics.MetricsAccumulator(usage="train", topK=5)


print(f"there are {len(dataloader)} batch to process")
it = iter(enumerate(dataloader))
while True:
    with prof.mesure("iterDataloader"):
        try: i, datas = next(it)
        except StopIteration:
            break
    
    with prof.mesure("print"):
        if i % 5 == 0:
            print(f"\rdoing: {i=}", end="", flush=True)
            
    with prof.mesure("split"):
        tokens: torch.Tensor = datas["tokens"].to(torch.int64)
        assert isinstance(tokens, torch.Tensor)
        dtIndexes: list[int] = datas["datasetIndex"].tolist()
        svgIndex: list[int] = datas["svgIndex"].tolist()
        chunckIndex: list[int] = datas["chunckIndex"].tolist()
        nbChars: int = sum([len(dataset.chunks[i].text) for i in dtIndexes])

    with prof.mesure("target"):
        if True:
            target = tokens
        else: target = torch.tensor(list(range(256)), dtype=torch.int64).view(1, -1)
        target = target.to(testDevice)

    with prof.mesure("logits"):
        batchSize, seq_len = target.shape
        vocab_size = tokenizer.get_vocab_size()
        if True:
            logits = torch.full((batchSize, seq_len, vocab_size), -10.0, device=testDevice)
            mask = (target != svg_dataset.IGNORE_INDEX)
            b, t = mask.nonzero(as_tuple=True)
            logits[b, t, target[b, t]] = 0.0
            logits += 3.0 * torch.randn(batchSize, seq_len, vocab_size, device=testDevice)
        else: logits = torch.randn(batchSize, seq_len, vocab_size, device=testDevice)# * 0.001

    # print(f"logits of shape: {logits.shape}[{logits.dtype}]")
    # print(f"target of shape: {target.shape}[{target.dtype}]")
    
    accum.batch_logits_metrics(logits, target, totalNbChars=nbChars)
    if i > 50:
        break

print()
res = accum.get_metrics()
prettyPrint(prof.pretty_totalTimes())
prettyPrint(accum._prof.pretty_totalTimes())

there are 432 batch to process
doing: i=50
{
    iterDataloader: 16.910 ms,
    split: 9.893 ms,
    target: 9.401 ms,
    logits: 65.305 ms,
    print: 7.144 ms
}
{
    loss1: 2.374 ms,
    loss2: 2.540 ms,
    filter: 379.485 ms,
    CE_related: 125.069 ms,
    entropy: 243.332 ms,
    SD: 33.659 ms,
    topK: 5.561 ms,
    accuacy: 294.693 ms
}


In [86]:
prettyPrint(res)

{
    CE_train: 1.7265773803450795,
    CE2_train: 0.7907985723572308,
    PPL_train: 5.621381096061931,
    PPL2_train: 2.205156700493593,
    BPC_train: 1.262000809743837,
    ENTROPY_train: 2.0390745459719324,
    LOGITS_SD_train: 3.0270969012059132,
    TOP-1_train: 0.5946615082118181,
    TOP-5_train: 0.8165130922572427
}


{
    loss1: 17.312 ms,
    loss2: 23.882 ms,
    filter: 3.057 sec,
    CE_related: 88.195 ms,
    entropy: 1.744 sec,
    SD: 285.440 ms,
    topK: 56.346 ms,
    accuacy: 2.536 sec
}

In [ ]:
# logits: (batch_size, context_size, vocab_size) | targets: (batch_size, context_size)
_, _, vocab_size = logits.shape
logits = logits[targets != IGNORE_INDEX].reshape((-1, vocab_size))
targets = targets[targets != IGNORE_INDEX]
# => logits: (nb_tokens, vocab_size) | targets: (nb_tokens)

In [255]:
accum = metrics.metrics.MetricsAccumulator(usage="train", topK=5)
t1 = time.perf_counter()
accum.batch_logits_metrics(logits, target, totalNbChars=int(1.97*seq_len*batchSize))
t2 = time.perf_counter()
prettyPrint(accum.get_metrics())
prettyPrint(accum._prof.pretty_totalTimes())
tt = sum(accum._prof.totalTimes().values())
print(prettyTime(tt), tt/(t2-t1), batchSize/tt)

{
    CE_train: 0.02616831287741661,
    PPL_train: 1.0265137094102275,
    BPC_train: 0.019163906201360277,
    CE2_train: 0.02616831287741661,
    PPL2_train: 1.0265137094102275,
    ENTROPY_train: 0.28445579528808596,
    LOGITS_SD_train: 0.4134490966796875,
    TOP-1_train: 1.0,
    TOP-5_train: 1.0
}
{
    loss1: 0.237 ms,
    loss2: 0.204 ms,
    CE_related: 11.272 ms,
    entropy: 13.949 ms,
    SD: 1.901 ms,
    topK: 0.273 ms,
    accuacy: 22.159 ms
}
49.996 ms 0.9841103485005943 1000.0736455503519


In [46]:
import torch
a = torch.arange(3*5*7).reshape((3,5,7))
b = (torch.randn((3, 5)) >= 0.7)
print(a)
print(b)
a2 = a[b].reshape((-1, 7))
print(a2)

tensor([[[  0,   1,   2,   3,   4,   5,   6],
         [  7,   8,   9,  10,  11,  12,  13],
         [ 14,  15,  16,  17,  18,  19,  20],
         [ 21,  22,  23,  24,  25,  26,  27],
         [ 28,  29,  30,  31,  32,  33,  34]],

        [[ 35,  36,  37,  38,  39,  40,  41],
         [ 42,  43,  44,  45,  46,  47,  48],
         [ 49,  50,  51,  52,  53,  54,  55],
         [ 56,  57,  58,  59,  60,  61,  62],
         [ 63,  64,  65,  66,  67,  68,  69]],

        [[ 70,  71,  72,  73,  74,  75,  76],
         [ 77,  78,  79,  80,  81,  82,  83],
         [ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]]])
tensor([[False,  True, False,  True,  True],
        [False, False, False,  True, False],
        [False, False, False,  True, False]])
tensor([[ 7,  8,  9, 10, 11, 12, 13],
        [21, 22, 23, 24, 25, 26, 27],
        [28, 29, 30, 31, 32, 33, 34],
        [56, 57, 58, 59, 60, 61, 62],
        [91, 92,

In [156]:
metrics.metrics.get_learning_rates(model)

{'lr_lm_head': 0.009797958971132713,
 'lr_embedding': 0.4898979485566357,
 'lr_value_embeds': 0.4898979485566357,
 'lr_residuals': 0.005,
 'lr_x0': 0.5,
 'lr_transformers_grp_0': 0.02,
 'lr_transformers_grp_1': 0.02,
 'lr_transformers_grp_2': 0.02,
 'lr_transformers_grp_3': 0.02}

In [ ]:
dataloader = DataLoader(dataset4096, batch_size=32, shuffle=True, num_workers=0)